# Adverse Drug Reaction (ADR) Prediction
**Department of Pharmacy — TMU**  
**Dataset:** Antiplatelet/Anticoagulant Patient Records  
**Task:** Binary Classification — ADR Present vs Absent  
**Model:** Logistic Regression (class_weight='balanced')

---

## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    f1_score, precision_score, recall_score, accuracy_score
)

print('Libraries loaded.')

## Step 2 — Load Dataset

In [ ]:
from google.colab import files
uploaded = files.upload()

import io
filename = list(uploaded.keys())[0]
df_raw = pd.read_excel(io.BytesIO(uploaded[filename]), sheet_name='Form responses 1')

print(f'Dataset loaded: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns')

## Step 3 — Data Overview

In [ ]:
print('=== Target Distribution (ADR) ===')
print(df_raw['ADR'].value_counts(dropna=False))

print('\n=== Null Counts ===')
print(df_raw.isnull().sum())

## Step 4 — Cleaning & Feature Engineering

In [ ]:
df = df_raw.copy()

# Normalise target
df['ADR'] = df['ADR'].str.strip().str.upper()
df = df[df['ADR'].isin(['PRESENT', 'ABSENT'])].copy()
df['ADR_BINARY'] = (df['ADR'] == 'PRESENT').astype(int)

# Age: extract numeric
df['AGE_CLEAN'] = df['AGE'].astype(str).str.extract(r'(\d+)').astype(float)

# Gender: binary
df['GENDER_BINARY'] = df['GENDER'].str.strip().str.upper().apply(
    lambda x: 1 if x == 'MALE' else 0
)

# Past medical history: has comorbidity?
df['HAS_COMORBIDITY'] = df['PAST MEDICAL HISTORY'].apply(
    lambda x: 0 if (pd.isna(x) or str(x).strip().upper() == 'NO') else 1
)

# Drug flags from antiplatelet column
ap_col = 'ANTI PLATELETS AGENTS PRESCRIBED '
df['HAS_ASPIRIN']     = df[ap_col].str.upper().str.contains('ASPIRIN',     na=False).astype(int)
df['HAS_CLOPIDOGREL'] = df[ap_col].str.upper().str.contains('CLOPIDOGREL', na=False).astype(int)
df['HAS_TICAGRELOR']  = df[ap_col].str.upper().str.contains('TICAGRELOR',  na=False).astype(int)

# Anticoagulant prescribed?
df['HAS_ANTICOAG'] = df['ANTICOAGULANT AGENTS PRESCRIBED'].notna().astype(int)

# Cardiac diagnosis flag
acs_keywords = ['ACS', 'AWMI', 'IWMI', 'NSTEMI', 'CORONARY', 'ANGINA', 'CAD', 'RWMI']
df['HAS_CARDIAC_DX'] = df['DIAGNOSIS'].apply(
    lambda x: 1 if any(k in str(x).upper() for k in acs_keywords) else 0
)

print('Cleaning complete.')
print(f'Records: {len(df)}')
print(f'ADR Distribution: {df["ADR_BINARY"].value_counts().to_dict()}')

## Step 5 — Feature Selection & Train/Test Split

In [ ]:
FEATURES = [
    'AGE_CLEAN',
    'GENDER_BINARY',
    'HAS_COMORBIDITY',
    'HAS_ASPIRIN',
    'HAS_CLOPIDOGREL',
    'HAS_TICAGRELOR',
    'HAS_ANTICOAG',
    'HAS_CARDIAC_DX'
]

df_model = df[FEATURES + ['ADR_BINARY']].dropna()
X = df_model[FEATURES]
y = df_model['ADR_BINARY']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features (required for Logistic Regression)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')
print(f'Train ADR distribution: {y_train.value_counts().to_dict()}')
print(f'Test  ADR distribution: {y_test.value_counts().to_dict()}')

## Step 6 — Train Logistic Regression
> `class_weight='balanced'` automatically adjusts weights inversely proportional to class frequency. This compensates for the imbalance (91% Absent, 9% Present) without oversampling.

In [ ]:
model = LogisticRegression(
    class_weight='balanced',
    random_state=42,
    max_iter=1000
)
model.fit(X_train_sc, y_train)
print('Model training complete.')

## Step 7 — Evaluation Metrics

In [ ]:
y_pred = model.predict(X_test_sc)
y_prob = model.predict_proba(X_test_sc)[:, 1]

print('=' * 50)
print('        CLASSIFICATION REPORT')
print('=' * 50)
print(classification_report(y_test, y_pred, target_names=['ADR Absent', 'ADR Present']))

print('=' * 50)
print('        KEY METRICS SUMMARY')
print('=' * 50)
print(f'Accuracy  : {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision : {precision_score(y_test, y_pred, zero_division=0):.4f}')
print(f'Recall    : {recall_score(y_test, y_pred, zero_division=0):.4f}')
print(f'F1 Score  : {f1_score(y_test, y_pred, zero_division=0):.4f}')
try:
    print(f'AUC-ROC   : {roc_auc_score(y_test, y_prob):.4f}')
except:
    print('AUC-ROC   : N/A')

## Step 8 — Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Predicted: Absent', 'Predicted: Present'],
    yticklabels=['Actual: Absent', 'Actual: Present'],
    linewidths=0.5, ax=ax
)
ax.set_title('Confusion Matrix — ADR Prediction', fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Actual Label', fontsize=12)
ax.set_xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 9 — ROC Curve

In [ ]:
try:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_score = roc_auc_score(y_test, y_prob)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(fpr, tpr, color='steelblue', lw=2, label=f'ROC Curve (AUC = {auc_score:.3f})')
    ax.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--', label='Random Classifier')
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate (Recall)', fontsize=12)
    ax.set_title('ROC Curve — ADR Prediction', fontsize=14, fontweight='bold')
    ax.legend(loc='lower right', fontsize=11)
    plt.tight_layout()
    plt.savefig('roc_curve.png', dpi=150, bbox_inches='tight')
    plt.show()
except Exception as e:
    print(f'ROC skipped: {e}')

## Step 10 — Odds Ratios (Clinical Interpretation)
> Odds ratios show how much each factor *increases or decreases* the likelihood of an ADR. Values > 1 increase risk, values < 1 decrease risk. This is the most clinically useful output for a pharmacy thesis.

In [ ]:
coef_df = pd.DataFrame({
    'Feature': FEATURES,
    'Coefficient': model.coef_[0]
})
coef_df['Odds_Ratio'] = np.exp(coef_df['Coefficient'])
coef_df = coef_df.sort_values('Odds_Ratio', ascending=True)

print('=== Odds Ratios per Feature ===')
print(coef_df[['Feature','Odds_Ratio']].to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#d9534f' if v > 1 else '#5bc0de' for v in coef_df['Odds_Ratio']]
ax.barh(coef_df['Feature'], coef_df['Odds_Ratio'], color=colors, edgecolor='white')
ax.axvline(x=1.0, color='black', linestyle='--', linewidth=1, label='No effect (OR=1)')
ax.set_title('Odds Ratios — ADR Risk Factors', fontsize=14, fontweight='bold')
ax.set_xlabel('Odds Ratio', fontsize=12)
ax.set_ylabel('Feature', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('odds_ratios.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nInterpretation: Red bars = increased ADR risk | Blue bars = decreased ADR risk')

## Step 11 — Download Outputs

In [ ]:
from google.colab import files
files.download('confusion_matrix.png')
files.download('roc_curve.png')
files.download('odds_ratios.png')

---
## Summary

| Item | Detail |
|------|--------|
| Model | Logistic Regression |
| Imbalance Handling | class_weight='balanced' |
| Split | 80% train / 20% test (stratified) |
| Primary Metric | Recall — critical for ADR detection |
| Key Clinical Output | Odds Ratios per feature |

> **Limitation:** With only ~22 ADR-positive cases in 272 records, the test set contains ~4 positive samples. Metric stability is limited by dataset size. Results should be interpreted alongside clinical domain knowledge rather than in isolation.